In [1]:
import sys
"""
!pip install --upgrade transformers sentence-transformers

!{sys.executable} -m pip install --upgrade --force-reinstall --no-cache-dir transformers sentence-transformers

 On met à jour Pandas pour qu'il s'aligne avec le reste
!{sys.executable} -m pip install --upgrade pandas "numpy<2.0"

!{sys.executable} -m pip install gensim
"""
import os
import re
import json
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# we use this ibrary to extract the text out of the pdf
from pypdf import PdfReader

# those are the libraries required for sbert, tf-idf and word2vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec

# (for text generation
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# used for distributed computing system. 
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession 
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    LongType, FloatType, ArrayType
)

# Configuration of the rag pipeline
Instead of scattering our parameters everywhere in the notebook, we centralize everything in one dataclass. This is easier and faster when you want to test another embedding strategy or change chunk size, because you only need to change your settings in this cell 

In [2]:
@dataclass
class RAGConfig:
    # paths for storing the pdf results of embeddings
    input_dir: str = "course_material"
    output_dir: str = "project/embeddings"
    parallelism: int = 4 
    # number of cpu cores used

    #  Chunking 
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # Retrieval 
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "tfidf"   # {"tfidf", "sbert", "word2vec"}

    #  Embedding models 
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    #  Generator (fallback: small, CPU-friendly) 
    gen_model_name: str = "google/flan-t5-small"
    max_new_tokens: int = 256

cfg = RAGConfig()

# Spark Initialization
We are using spark to treat several pdf's in parallel thanks of its processes which optimizes the use of cpu cores.

In [5]:
# Init Park
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("SparkContext existant arrêté.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

SparkContext existant arrêté.
Spark version: 3.5.0


# loading pdf's
we retrieve the list with every pdf files in the configured folder, then we give it to spark in rdd. Every element of the RDD is a path to a pdf. Spark will then treat then in parallel.

In [4]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

#  quick sanity check (should not error once implemented) 
pdf_paths = get_paths_pdf_files(cfg.input_dir)
rdd = sc.parallelize(pdf_paths, numSlices=cfg.parallelism)
print(pdf_paths)

[]


# Page by page extraction of the text
we define the function wich will open a pfd and extract its text. This function will be applied in parallel on every pdf thanks to spark.

The flatmap at the end flattends the list of pages of each pdf in one RDD where each element is one individual page

In [ ]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    
    import logging
    logging.getLogger("pypdf").setLevel(logging.ERROR)
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

pages_rdd = rdd.flatMap(extract_pages_from_pdf)

# Cleaning of the text
pypdf reconstruct the text line by line from the pdf. This creates artefacts (words that are cut at the end of a line, line jumps in the middle of a sentence, multiple spaces between some words).
We introduce 4 cleaning steps:
1) pypdf splits words at the end of a line with a hyphen
2) it replaces the remaining line breaks with space
3) then it reduce multiple spaces to a single space
4) finally it remove leading and trailing spaces

In [ ]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for PDF-extracted text."""
    text = page_from_pdf["text"]
    # 1)
    text = text.replace("-\n", "")
    
    # 2)
    text = text.replace("\n", " ")
    
    # 3)
    text = re.sub(r"\s+", " ", text)
    
    # 4)
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map = application of the cleaning process on every page
clean_rdd = pages_rdd.map(clean_text)
print(clean_rdd.count(), "pages extraites")
print(clean_rdd.first())

# Splitting into chunks
each chunk is a rolling window of a number (chunk_size_words) of words. The overlap_words guarantees that if a concept is on more than one chunk, it won't be lost. Each chunk keeps note of its position in the page to be able to find back its source.

In [ ]:
def word_based_chunk_page(page_from_pdf: dict) -> list:
    """Split one page into overlapping word chunks."""
    
    chunk_size_words = cfg.chunk_size_words 
    overlap_words = cfg.chunk_overlap_words

    words = page_from_pdf["text"].split()
    page_num = page_from_pdf["page_num"]
    
    chunks = []
    step = chunk_size_words - overlap_words
    #step = distance between the start of 2 consecutive chunks
    
    if step <= 0:
        raise ValueError("overlap_words must be smaller than chunk_size_words")
    
    chunk_index = 0
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_index}",
            "pdf_name": page_from_pdf["pdf_name"],
            "page_num":  page_num,
            "text": " ".join(chunk_words),
            "start_word": start,        
            "end_word": end - 1
        })
        chunk_index += 1
        if end >= len(words):
            break
    return chunks
# build chunks
# flatmap allows every page to become a list of chunks, we flatten it in one RDD
clean_chunk_rdd = clean_rdd.flatMap(word_based_chunk_page)
print(clean_chunk_rdd.count(), " chunks")
print(clean_chunk_rdd.first())

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Creation of the embeddings pipeline
We setup a pipeline for our embeddings. We transform every text chunk in embedding vectors so that they can be processed y transformers later.

We use TF-IDF, SBERT and WOrd2Vec. We create a function sfg.embedding_strategy for our common pipeline.

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, FloatType, ArrayType
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, Word2Vec as SparkWord2Vec
from pyspark.sql.functions import udf, lit

# 1. we reuse the sparkcontext that we created before
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

# 2. we introduce zipWithIndex which add a unique identifier to every chunk 
#    then we create the dataframe with the id's and metadata
rdd_base = clean_chunk_rdd.zipWithIndex().map(
    lambda x: (
        x[1],                        # id (LongType)
        x[0]["pdf_name"],            # source_pdf
        x[0]["page_num"],            # page_num
        x[0]["chunk_id"],            # chunk_id
        x[0]["start_word"],          # start_word (Entier)
        x[0]["end_word"],            # end_word (Entier)
        x[0]["text"],                # chunk_text
    )
)

schema_base = StructType([
    StructField("id", LongType(), False),
    StructField("source_pdf", StringType(), True),
    StructField("page_num", IntegerType(), True),
    StructField("chunk_id", StringType(), True),
    StructField("start_word", IntegerType(), True),
    StructField("end_word", IntegerType(), True),
    StructField("chunk_text", StringType(), True),
])

df_base = spark.createDataFrame(rdd_base, schema=schema_base)

# final schema with embeddings columns
final_schema = StructType(schema_base.fields + [
    StructField("embedding", ArrayType(FloatType()), True),
    StructField("embedding_strategy", StringType(), True)
])

df_final = None # 


# 3. choice between the different embedding strategies


if cfg.embedding_strategy == "sbert":
    # 1ST STRATEGY : SBERT 
    def embed_partition_sbert(partition):
        from sentence_transformers import SentenceTransformer
        import numpy as np
        
        model = SentenceTransformer(cfg.sbert_model_name)
        for row in partition:
            row_dict = row.asDict()
            emb = model.encode(row_dict["chunk_text"], normalize_embeddings=True)
            row_dict["embedding"] = np.asarray(emb, dtype=np.float32).tolist()
            row_dict["embedding_strategy"] = "sbert"
            yield row_dict

    rdd_embedded = df_base.rdd.mapPartitions(embed_partition_sbert)
    df_final = spark.createDataFrame(rdd_embedded, schema=final_schema)

elif cfg.embedding_strategy == "tfidf":
    # 2ND STRATEGY : TF-IDF 
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000)
    featurizedData = hashingTF.transform(wordsData)
    
    idf = IDF(inputCol="rawFeatures", outputCol="features")
    idfModel = idf.fit(featurizedData)
    rescaledData = idfModel.transform(featurizedData)
    ## Spark stores vectors in sparse format, then we convert them to Python lists
    # so we can save them in parquet format and easily read them back into Notebook 2
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = rescaledData.withColumn("embedding", vector_to_array("features")) \
                           .withColumn("embedding_strategy", lit("tfidf")) \
                           .drop("words", "rawFeatures", "features")
    idfModel.write().overwrite().save("/home/guest/shared/course_material/tfidf_model.model")

elif cfg.embedding_strategy == "word2vec":
    # 3RD STRATEGY : Word2Vec 
    tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
    wordsData = tokenizer.transform(df_base)
    
    word2Vec = SparkWord2Vec(vectorSize=100, minCount=2, inputCol="words", outputCol="features")
    model = word2Vec.fit(wordsData)
    w2vData = model.transform(wordsData)
    
    vector_to_array = udf(lambda v: v.toArray().tolist(), ArrayType(FloatType()))
    
    df_final = w2vData.withColumn("embedding", vector_to_array("features")) \
                      .withColumn("embedding_strategy", lit("word2vec")) \
                      .drop("words", "features")

    model.write().overwrite().save("/home/guest/shared/course_material/w2v_model.model")
    

# 4. BACKUP AND TEST

if df_final is not None:
    import os
    
    # 1. On sauvegarde d'abord dans un dossier 100% Linux (pas de conflit avec ton PC)
    linux_internal_path = f"/home/guest/embeddings_{cfg.embedding_strategy}.parquet"
    df_final.write.mode("overwrite").parquet(linux_internal_path)
    print(f" Sauvegarde réussie dans le conteneur : {linux_internal_path}")
    
    # 2. On copie manuellement le résultat fini vers ton dossier partagé
    shared_path = f"/home/guest/shared/course_material/embeddings_{cfg.embedding_strategy}.parquet"
    os.system(f"cp -r {linux_internal_path} {shared_path}")
    print(f" Fichiers copiés vers ton PC : {shared_path}\n")
    
    # 3. Test de lecture (depuis le dossier partagé)
    print("="*70)
    print("VÉRIFICATION : LECTURE DU FICHIER PARQUET GÉNÉRÉ")
    print("="*70)
    test_df = spark.read.parquet(shared_path)
    print(f"Nombre de chunks sauvegardés et relus : {test_df.count()}")
    test_df.select("id", "source_pdf", "chunk_id", "start_word", "end_word", "embedding_strategy").show(3)

NameError: name 'clean_chunk_rdd' is not defined

# Inspection of the parquet file
We list and manually inspect

In [ ]:
import os
chemin = f"/home/guest/shared/course_material/embeddings_{cfg.embedding_strategy}.parquet"
print("Fichiers trouvés dans le dossier :")
print(os.listdir(chemin))

In [ ]:
print("ok")

# Inspecting the generated chunks  

In [ ]:
# On relit le parquet avec Spark
df_test = spark.read.parquet("/home/guest/embeddings_sbert.parquet")

# On prend juste 5 lignes, on sélectionne quelques colonnes utiles, et on le convertit en Pandas pour un bel affichage
affichage = df_test.select("source_pdf", "page_num", "chunk_text").limit(5).toPandas()

# On affiche le tableau
display(affichage)